# AlphaFold2 Structure Prediction

This notebook demonstrates protein structure prediction using AlphaFold2 via the ColabDesign JAX wrapper. We cover three common workflows with `run_alphafold2`: single-sequence prediction without MSA for fast screening, multi-chain complex prediction using the Multimer extension, and MSA-assisted prediction that leverages evolutionary information from ColabFold search for higher accuracy.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("alphafold2")
display_overview("alphafold2")
display_docs_section("alphafold2", "Background")

# AlphaFold2

> AlphaFold2 predicts 3D protein structures from amino acid sequences using the original DeepMind model via the ColabDesign JAX wrapper (`alphafold2-prediction`). It supports optional [multiple sequence alignment](https://en.wikipedia.org/wiki/Multiple_sequence_alignment) (MSA) generation via ColabFold search for improved accuracy, and can predict both monomeric and multimeric protein structures.

## Background

**What does this tool measure/predict?**
AlphaFold2 predicts the 3D atomic coordinates of protein structures from amino acid sequences. It outputs full-atom protein structures with per-residue confidence scores (pLDDT), global structure quality metrics (pTM), and inter-chain confidence for multimers (ipTM).

**Why is this important?**

* Protein engineering: validate that designed sequences fold into intended structures
* Drug discovery: predict target protein structures for virtual screening
* Functional annotation: infer function from predicted 3D structure
* Protein design pipelines: final quality gate before experimental validation
* Structural biology: generate models for proteins without experimental structures

**Scientific foundation:**
AlphaFold2 uses a two-track architecture combining evolutionary and structural information:

1. **Multiple Sequence Alignment (MSA) processing:** Evolutionary information from homologous sequences is processed through the Evoformer module, which uses axial attention to capture coevolutionary patterns that constrain 3D structure.
2. **Structure module:** Iteratively refines 3D coordinates using invariant point attention (IPA), which operates directly in 3D space respecting rotational and translational symmetry.
3. **Recycling:** The prediction is iteratively refined by feeding outputs back through the network (configurable via `num_recycles`), progressively improving accuracy.
4. **Ensemble averaging:** Multiple independently trained model parameter sets (1-5) can be averaged for higher confidence predictions.

The model was trained on experimentally determined structures from the [Protein Data Bank](https://www.rcsb.org/) and achieves near-experimental accuracy for many protein families.

## Available tools

In [2]:
display_available_tools("alphafold2")

- **`run_alphafold2()`** — Protein structure prediction using AlphaFold2 via ColabDesign

### `run_alphafold2`

AlphaFold2 predicts protein 3D structures from amino acid sequences using a deep learning architecture that combines multiple sequence alignments, pairwise distance representations, and recycling iterations to generate near-experimental-accuracy coordinates. The model supports single-chain proteins and multi-chain complexes via the Multimer extension. Optional MSA generation via ColabFold search provides evolutionary context from homologous sequences, substantially improving confidence scores. The `num_recycles` parameter controls how many times predictions are iteratively refined, and the `model_num` parameter selects among five independently trained parameter sets.

In [3]:
from pathlib import Path
from proto_tools import AlphaFold2Input, AlphaFold2Config, run_alphafold2

In [4]:
# Display input docs
display_api_reference("alphafold2", "input", "run_alphafold2")

# Aequorea victoria GFP (UniProt P42212) — 238-residue β-barrel fluorescent
# protein, classic folding benchmark with dense MSA coverage. Reused below
# in the MSA-assisted example.
protein_sequence = (
    "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF"
    "ICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGY"
    "VQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILG"
    "HKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQ"
    "NTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGIT"
    "HGMDELYK"
)
inputs = AlphaFold2Input(complexes=[protein_sequence])

**Input** — `AlphaFold2Input`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `complexes` | `List[StructurePredictionComplex]` | required | List of complexes to predict structures for. Inherited from `StructurePredictionInput`. Each complex can contain one or more protein chains. |
| `chains` | `List[Chain]` | required | Chains in the complex. Each chain can be: |
| `msas` | `Dict[string, MSA]` |  | Pre-computed MSAs keyed by protein sequence. Populated by preprocess() or supplied directly. Default: None. |

In [5]:
# Display config docs
display_api_reference("alphafold2", "config", "run_alphafold2")

# Configure AlphaFold2
config = AlphaFold2Config(
    use_msa=True,
    num_recycles=3,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `AlphaFold2Config`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `num_recycles` | `integer` | `3` | Number of recycling iterations through the model. Higher values can improve accuracy at the cost of computation time. |
| `model_num` | `integer` | `1` | Which AlphaFold2 model parameter set to use (1-5). AF2 ships 5 independently trained parameter sets. Different sets can produce different predictions. Mutually exclusive with `num_ensemble_models > 1`; set one or the other. Default: 1. |
| `num_ensemble_models` | `integer` | `1` | Number of model parameter sets to run and average. Running multiple models and averaging their outputs can improve prediction quality at the cost of increased computation time. Mutually exclusive with `model_num`; when ensembling, models are selected from the full pool (models 1 through N). Range: 1-5. Default: 1. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `use_msa` | `boolean` | `True` | Whether to generate and use Multiple Sequence Alignments (MSAs) for protein chains using ColabFold search. Inherited from `MSAStructurePredictionConfig`. Default: `True`. |
| `verbose` | `boolean` | `False` | Whether to print status messages during execution. Inherited from `BaseConfig`. Default: `False`. |
| `device` | `string` | `cuda` | Device to run the model on (`"cuda"`, `"cpu"`). Inherited from `StructurePredictionConfig`. Default: `"cuda"`. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |
| `colabfold_search_config` | `ColabfoldSearchConfig` |  | Configuration for ColabFold MSA search. Only used when `use_msa=True`. Inherited from `MSAStructurePredictionConfig`. Default: `None`. |

In [6]:
# Run structure prediction
result = run_alphafold2(inputs, config)

Running run_colabfold_search [00:00]

Folding structures (AlphaFold2):   0%|          | 0/1 [00:00<?, ?structure/s]

In [7]:
# Display output docs
display_api_reference("alphafold2", "output", "run_alphafold2")

structure = result.structures[0]

# Print confidence metrics
print(f"  Length: {structure.num_residues} residues")
print(f"  Average pLDDT: {structure.metrics.avg_plddt:.3f}")
print(f"  pTM score: {structure.metrics.ptm:.3f}")

**Output** — `AlphaFold2Output`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `structures` | `List[Structure]` | required | Predicted structures with confidence metrics. |
| `structure` | `string` | required | Raw structure content in PDB or CIF format. |
| `structure_format` | `string` |  | Format of the content string (auto-detected if omitted). |
| `b_factor_type` | `BFactorType` |  | What the B-factor column represents. |
| `source` | `string` |  | Optional source identifier (filepath or tool name). |
| `metrics` | `StructureMetrics` |  | Associated metrics (e.g., pLDDT, pTM scores, per-chain lists, pairwise matrices). None values are stripped at construction. |

  Length: 238 residues
  Average pLDDT: 0.947
  pTM score: 0.890


In [8]:
# Visualize predicted structure (auto-coloured by pLDDT)
structure.visualize()

#### Multi-chain complex prediction

When a complex contains more than one chain, AlphaFold2 automatically uses the Multimer model parameters. The iPTM score provides an additional measure of interface quality beyond the global pTM, which is especially useful for evaluating whether the predicted inter-chain contacts are reliable.

In [9]:
# Two-chain protein complex
chain_a = "MARFLGLGARYTWM"
chain_b = "YTWHKLARFGMVLS"

# Create multi-chain input
inputs_complex = AlphaFold2Input(complexes=[[chain_a, chain_b]])

# Configure (multimer mode is automatic when >1 chain)
config_complex = AlphaFold2Config(
    use_msa=False,
    num_recycles=3,
    device="cuda",  # Change to "cpu" if no GPU available
)

# Run prediction
result_complex = run_alphafold2(inputs_complex, config_complex)
complex_structure = result_complex.structures[0]

# Print metrics (iPTM is available for multimer)
print(f"  Chains: {complex_structure.num_chains}")
print(f"  Average pLDDT: {complex_structure.metrics.avg_plddt:.3f}")
print(f"  pTM score: {complex_structure.metrics.ptm:.3f}")
print(f"  iPTM score: {complex_structure.metrics.get('iptm', 'N/A')}")

# Visualize the complex with each chain in a distinct colour
complex_structure.visualize(color_by="chain")

Folding structures (AlphaFold2):   0%|          | 0/1 [00:00<?, ?structure/s]

  Chains: 2
  Average pLDDT: 0.440
  pTM score: 0.111
  iPTM score: 0.018126871436834335


## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD. The B-factor column in exported files contains normalized pLDDT confidence scores (0.0 to 1.0), so coloring by B-factor in a viewer directly shows model confidence per residue.

In [10]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to PDB files
result.export(name="alphafold2_structure", export_path=output_dir, file_format="pdb")
print(f"Structure exported to {output_dir / 'alphafold2_structure.pdb'}")

Structure exported to example_output/alphafold2_structure.pdb
